# ENES-LifeWatch User Story

This notebook demonstrates the possibility of linking data from the climate domain (ENES catalog) with data from the life sciences domain (LifeWatch catalog) to investigate, for example, how biodiversity observations relate to climatic.conditions. 

More specifically, the workflow combines biodiversity observations with climate data, producing an enriched dataset ready for analysis. The notebook also visualises the observations together with climate layers, demonstrating the integration of data from the LifeWatch ERIC Metadata Catalogue and the ENES Thematica Data Catalogue within a single workflow.

In [ ]:
import xarray as xr
import pandas as pd
import dask
dask.config.set({"array.slicing.split_large_chunks": False})
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
cmip_path = "https://object-store.brno.openstack.cloud.e-infra.cz/swift/v1/KEY_db4f6691a81f49d0a8dda59b26259b5b/vo.enes.org-cmip6/CMIP6.CMIP.CMCC.CMCC-ESM2.historical.r1i1p1f1.day.tas.gn/"
lifewatch_path = "./amphipod_species_EU_marine_waters.csv"

### Access to climate data

In [ ]:
ds_cmip = xr.open_zarr(
    cmip_path,
    consolidated=True
)

### Spatial and temporal subset

In [ ]:
ds_cmip = ds_cmip.convert_calendar("standard")

year = 2000

# Europe bounding box 
lon_min, lon_max = -25, 45
lat_min, lat_max = 34, 72

# Coordinate normalization
ds_cmip = ds_cmip.assign_coords(
    lon=((ds_cmip.lon + 180) % 360) - 180
).sortby("lon")

# Data subset
temp_eu = ds_cmip["tas"].sel(
    time=str(year)
).mean("time").sel(
    lon=slice(lon_min, lon_max),
    lat=slice(lat_min, lat_max)
)

In [ ]:
ds_cmip

### Access to biodiversity data

In [ ]:
df = pd.read_csv(lifewatch_path)
df

### Spatial and temporal subset

In [ ]:
df["eventDate_single"] = (
    df["eventDate"]
    .astype(str)
    .str.split("/")
    .str[0]
)

df["eventDate_single"] = pd.to_datetime(df["eventDate_single"])
occ = df[df["eventDate_single"].dt.year == year]

In [ ]:
occ_eu = occ[
    (occ["decimalLongitude"] >= lon_min) &
    (occ["decimalLongitude"] <= lon_max) &
    (occ["decimalLatitude"] >= lat_min) &
    (occ["decimalLatitude"] <= lat_max)
]

### Data visualization

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig = plt.figure(figsize=(12, 9), dpi=300)

ax = plt.axes(projection=ccrs.PlateCarree())

mesh = ax.pcolormesh(
    temp_eu.lon,
    temp_eu.lat,
    temp_eu.values,
    cmap="RdYlBu_r",
    shading="auto",
    alpha=0.85,
    transform=ccrs.PlateCarree()
)

cbar = plt.colorbar(
    mesh,
    ax=ax,
    shrink=0.75,
    pad=0.02
)

cbar.set_label("Mean annual temperature (°C)")

# ---------------------------
# Occurrences
# ---------------------------

ax.scatter(
    occ_eu["decimalLongitude"],
    occ_eu["decimalLatitude"],
    s=10,
    c="black",
    edgecolors="white",
    linewidths=0.3,
    alpha=0.75,
    transform=ccrs.PlateCarree(),
    zorder=10
)

# ---------------------------
# Basemap
# ---------------------------

ax.add_feature(cfeature.COASTLINE, linewidth=0.6)
ax.add_feature(cfeature.BORDERS, linewidth=0.4)

ax.set_extent(
    [lon_min, lon_max, lat_min, lat_max],
    crs=ccrs.PlateCarree()
)

# ---------------------------
# Grid
# ---------------------------

gl = ax.gridlines(
    draw_labels=True,
    linewidth=0.3,
    color="gray",
    alpha=0.5,
    linestyle="--"
)

gl.top_labels = False
gl.right_labels = False

# ---------------------------
# Title
# ---------------------------

plt.title(f"Temperature and species occurrences ({year})", fontsize=16)
plt.tight_layout()

plt.show()

In [ ]:
fig.savefig("./temperature_and_species_occurrences_2000.png", dpi=300, bbox_inches="tight")